In [ ]:
#Define virtual stations location and initial number of emoped for specific users scenario

In [ ]:
### Imports 
import os
import math
import sys
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D
import matplotlib.patches as patches

import pandas as pd
import re
import xml.etree.ElementTree as ET
import json

sys.path.append('../../')
from script.conversion.bison.coordinates import rd_to_utm
from mnms.graph.layers import PublicTransportLayer, MultiLayerGraph, OriginDestinationLayer
from mnms.generation.roads import generate_pt_line_road, generate_one_zone
from mnms.generation.layers import generate_bbox_origin_destination_layer
from mnms.vehicles.veh_type import Tram, Metro, Bus
from mnms.generation.zones import generate_one_zone
from mnms.mobility_service.public_transport import PublicTransportMobilityService
from mnms.time import TimeTable, Dt, Time
from mnms.io.graph import load_graph, save_graph, save_odlayer
from mnms.tools.render import draw_roads, draw_line, draw_odlayer
from mnms.tools.geometry import points_in_polygon, get_bounding_box

In [ ]:
### Parameters

nb_emoped = 30

d_max = 1e3 # , max distance between hex and node for station

REF_X = 632e3
REF_Y = 5.8065e6

# Files and directories
current_dir = os.getcwd()
indir = current_dir + '/inputs/'
outdir = current_dir + '/outputs/'

f = open('params.json')
params = json.load(f)

#amsterdam_json_filepath = indir + 'new_network.json' # mlgraph with the road network only
#ams_dmd_path = indir + 'test_all_in_highway_7h_9h.csv' # dmd inside ringroad 7-9am
init_emoped_path = indir + 'init_pos_emoped.csv'
fn_stations = indir + 'emoped_stations'+str(nb_emoped)+'.csv'

np.random.seed(68568)

In [ ]:
### Get the MLGraph without TCs 
amsterdam_graph = load_graph(indir+params['fn_network'])
roads = amsterdam_graph.roads

# Based on demand data

In [ ]:
# Load dmd
db_dmd = pd.read_csv(indir+params['fn_demand'], sep=';')

In [ ]:
db_dmd

In [ ]:
nb_dmd = len(db_dmd)
origins = np.zeros((len(db_dmd),2))
destinations = np.zeros((len(db_dmd),2))
for i, row in db_dmd[:].iterrows():
    origins[i] = [float(o) for o in row['ORIGIN'].split(' ')]
    destinations[i] = [float(d) for d in row['DESTINATION'].split(' ')]

In [ ]:
nodes_pt = [k for k in roads.nodes.keys() if ('METRO' in k) or ('TRAM' in k)]
nodes_pt_pos = np.array([roads.nodes[k].position for k in nodes_pt])

In [ ]:
nb_pt_pts = len(nodes_pt_pos)


In [ ]:
orig_dest = list(origins)+list(destinations)+list(nodes_pt_pos)

In [ ]:
orig_dest[-1]

In [ ]:
# Associate closest node
nodes_id = [k for k in roads.nodes.keys() if not ('TRAM' in k or 'BUS' in k or 'METRO' in k)]
nodes_pos = np.array([roads.nodes[k].position for k in nodes_id])

closest_nodes = []
x_nodes = []
y_nodes = []
for (x,y) in orig_dest:
    dist_nodes = (x-nodes_pos[:,0])**2 + (y-nodes_pos[:,1])**2
    i_min = np.argmin(dist_nodes)
    x_nodes.append(nodes_pos[i_min,0])
    y_nodes.append(nodes_pos[i_min,1])
    if dist_nodes[i_min]>d_max**2:
        closest_nodes.append('to_delete')
    else:
        closest_nodes.append(nodes_id[i_min])


df_sta_dmd = pd.DataFrame({'closest_node':closest_nodes, 'x_node':x_nodes, 'y_node':y_nodes,
                           'type':['origin']*nb_dmd+['destination']*nb_dmd+['pt']*nb_pt_pts,
                          'ref_dmd':np.ones(2*nb_dmd+nb_pt_pts)})

print(sum(df_sta_dmd.closest_node=='to_delete'), 'stations too far')
df_sta_dmd = df_sta_dmd[df_sta_dmd.closest_node!='to_delete']
df_sta_dmd.reset_index(drop=True, inplace=True)

# Remove doppel station
a = df_sta_dmd['closest_node'].value_counts()
dbl_id = a[a>1].keys()
dmd_prob = []
for n_id in dbl_id:
    c=0
    st_id0 = df_sta_dmd[df_sta_dmd.closest_node == n_id].index[0]
    if df_sta_dmd.loc[st_id0, 'type']=='origin':
        c+=1
    for st_id in df_sta_dmd[df_sta_dmd.closest_node == n_id].index[1:]:
        if df_sta_dmd.loc[st_id, 'type']=='origin':
            c+=1
        df_sta_dmd.drop(st_id, inplace=True)
    c = min(c,10)
    df_sta_dmd.loc[st_id0,'ref_dmd']=c
df_sta_dmd.reset_index(drop=True, inplace=True)

for i in range(len(df_sta_dmd)):
    if df_sta_dmd.loc[i,'ref_dmd']==0 and df_sta_dmd.loc[i,'type'] in ['origin', 'pt']:
        df_sta_dmd.loc[i,'ref_dmd']=1

#for i in range(len(df_sta_dmd)):
#    if df_sta_dmd.loc[i,'type'] =='pt':
#        df_sta_dmd.loc[i,'ref_dmd']=1
#    else:
#        df_sta_dmd.loc[i,'ref_dmd']=0

In [ ]:
df_sta_dmd

In [ ]:
df_sta_dmd.ref_dmd.sum()

In [ ]:
#stations_minus = [ 22,  27,  38,  58,  71,  78,  81,  99, 129, 139, 147, 148, 161,
#       177, 188, 200, 209, 216, 220, 264, 274, 276, 298, 303, 305, 324,
#       354, 355, 364, 377, 399, 410, 433, 436, 441, 448, 453, 481, 517,
#       521, 531, 541, 587, 590, 600, 634, 720, 760]

In [ ]:
#df_sta_dmd.loc[stations_minus]

In [ ]:
# Generate initial positions
list_nb_emoped = np.zeros(len(df_sta_dmd), dtype=int)
#for _ in range(nb_emoped):
#    i = np.random.choice(df_sta_dmd.index, p = df_sta_dmd.ref_dmd.values/df_sta_dmd.ref_dmd.sum())
#    list_nb_emoped[i] += 1
dist2 = [(x - REF_X)**2 + (y - REF_Y)**2 for (x,y) in zip(df_sta_dmd.x_node, df_sta_dmd.y_node)]
i_sta = np.argmin(dist2)
list_nb_emoped[i_sta] = nb_emoped
df_sta_dmd['nb_emoped'] = list_nb_emoped

In [ ]:
df_sta_dmd

In [ ]:
plt.hist(df_sta_dmd.nb_emoped.values)

In [ ]:
fig, ax = plt.subplots(figsize=(15, 15))
draw_roads(ax, roads, color='grey', linkwidth=0.1, nodesize=0, draw_stops=False, node_label=False)

#plt.plot(df_sta.hex_x, df_sta.hex_y,'gd',alpha=0.1)
#plt.scatter(df_sta.x_node, df_sta.y_node, c=df_sta.nb_emoped)
#plt.scatter(df_sta_vip.x_node, df_sta_vip.y_node, c='r', marker='*', alpha=0.5)

plt.scatter(df_sta_dmd.x_node, df_sta_dmd.y_node,marker='+', c=df_sta_dmd.nb_emoped)



plt.scatter(REF_X, REF_Y, marker='o')

In [ ]:
## Save file 
df_sta_dmd.to_csv(fn_stations)